# **Práctica 4:**
# **Paralelismo a nivel de hilos: Paralelización mediante OpenMP y programación asíncrona del análisis forense de manipulación de imágenes digitales**

## Miembros de equipo:
## *Nicolás Grima Hernández* y *Miguel Perez Alonso*

#### **Tarea 0.1 Entrenamiento previo OpenMP:**

**0.1.1. ¿Para qué sirve la variable chunk?**

La variable chunk determina la granularidad del reparto de carga en una construcción de bucle paralelo. En este código específico, con un CHUNKSIZE de 100, el runtime de OpenMP descompone el espacio de iteraciones en paquetes (bloques) de 100 unidades. Estos paquetes son las unidades mínimas de trabajo que se asignan a cada hilo del equipo (thread team), permitiendo un control fino sobre el balanceo de carga frente a la sobrecarga
de gestión.


**0.1.2 Explica completamente el pragma:**

*#pragma omp parallel shared(a,b,c,chunk) private(i)*

La directiva #pragma omp parallel define una región paralela, activando el modelo de ejecución Fork-Join. Cuando el hilo maestro encuentra esta directiva, crea un equipo de hilos que ejecutan concurrentemente el bloque de código asociado.

- ¿Por qué y para qué se usa *shared(a,b,c,chunk)* en este
programa?

*shared(a, b, c, chunk)* establece que estas variables residen en el espacio de memoria global accesible por todos los hilos.
Las variables a, b y c se definen como compartidas para permitir el acceso concurrente a las estructuras de datos de entrada y asegurar que los resultados calculados persistan en la memoria principal tras la finalización de la región paralela.
La variable chunk es compartida porque actúa como un parámetro de configuración de solo lectura. No requiere copias locales, ya que su valor permanece constante y es utilizado por todos los hilos en la lógica de planificación.

- ¿Por qué la variable i está etiquetada como private en el
pragma?

*private(i)* es una cláusula fundamental para la ejecución correcta del programa. Al declararla como privada, cada hilo dispone de su propia instancia de la variable i en su pila local (stack).
Si la variable i fuera compartida, se produciría una condición de carrera o race condition, ya que múltiples hilos intentarían modificar simultáneamente el mismo contador. Esto provocaría accesos incorrectos a los índices de los vectores, resultados corruptos y comportamientos indeterminados.



**0.1.3 ¿Para qué sirve *schedule*? ¿Qué otras posibilidades hay?**

Lo que hace la *schedule* es definir el algoritmo de asignación de iteraciones a los hilos de ejecución. En nuestro código, schedule(dynamic, chunk) implementa una planificación bajo demanda donde cada hilo solicita un nuevo bloque de tamaño chunk conforme termina el anterior. Esta cláusula se muestra como la estrategia óptima para mitigar el desequilibrio de carga (load imbalance) cuando las iteraciones tienen costes computacionales diferentes.

Alternativas posibles de planificación en OpenMP:

· *static* (estática): El espacio de iteraciones se divide y asigna de forma determinista antes de la ejecución. Minimiza la sobrecarga (overhead) de gestión, pero es sensible a desequilibrios de carga. 

· *guided* (guiada): Los hilos reciben bloques de tamaño decreciente. Combina la baja sobrecarga inicial con un refinamiento del balanceo al final del bucle.

**0.1.4. ¿Qué tiempos y otras medidas de rendimiento podemos medir en secciones de código paralelizadas con OpenMP?**

Para caracterizar el comportamiento de un código paralelizado, se emplean las siguientes métricas fundamentales:
- *Tiempo de ejecución (Wall-clock time)*: Representa el tiempo real total transcurrido desde el inicio hasta el fin del programa. En entornos OpenMP, se mide con precisión mediante la función omp_get_wtime(). Es la métrica base sobre la cual se calculan todas las demás.
- *Speed-up*: Es el cociente entre el tiempo de ejecución secuencial y el tiempo de ejecución paralelo con P hilos. Esta métrica cuantifica cuántas veces más rápido es el programa tras la optimización. Un Speed-up igual al número de hilos se considera una aceleración lineal ideal.
- *Eficiencia*: Es la relación entre el Speed-up obtenido y el número de hilos utilizados. Se expresa habitualmente en porcentaje y mide el grado de aprovechamiento de los recursos del sistema. Una eficiencia del 100% indicaría que no hay pérdida de rendimiento por gestión de hilos.
- *Escalabilidad*: Evalúa cómo responde el sistema ante el incremento de recursos. *Strong Scaling* (escalabilidad fuerte) analiza la capacidad de reducir el tiempo de respuesta para un problema de tamaño constante al aumentar el número de hilos; *Weak Scaling* (escalabilidad débil) evalúa la capacidad de mantener un tiempo de ejecución constante cuando el tamaño del problema aumenta en la misma proporción que el número de hilos.
- *Sobrecarga (Overhead)*: Se refiere al tiempo adicional que consume la CPU en tareas que no son de cálculo directo, como la creación de hilos, la sincronización en barreras, el balanceo de carga y la comunicación. Un elevado overhead limita directamente el Speed-up máximo alcanzable.

#### **Tarea 0.2: Entrenamiento previo std::async**

**0.2.1 ¿Para qué sirve el parámetro *std::launch::async?***

El parámetro *std::launch::async* es una política de lanzamiento que indica explícitamente al runtime de C++ que la tarea debe ejecutarse de forma asíncrona en un hilo nuevo y separado de forma inmediata. Al utilizar esta política, se garantiza el solapamiento de la ejecución de las funciones, permitiendo que tareas independientes progresen simultáneamente. En el código de ejemplo, esto permite que mientras una tarea está en estado de espera (sleep), la otra pueda estar utilizando ciclos de CPU o su propia cuenta atrás, reduciendo el tiempo total de ejecución.

En el código del ejemplo, gracias a *std::launch::async*, las tareas 1 y 2 se ejecutan simultáneamente. Por eso, aunque sumen 5 segundos de trabajo (2000ms + 3000ms), el programa terminará en aproximadamente 3 segundos (el tiempo de la tarea más larga).

**0.2.2 Calcula el tiempo que tarda el programa con *std::launch::async* y
*std::launch::deferred*. ¿A qué se debe la diferencia de tiempos?**

- *Tiempo con std::launch::async*: Aproximadamente 3000 ms. Dado que las tareas se ejecutan en paralelo, el tiempo total de respuesta está limitado por la tarea de mayor duración (la Tarea 2, de 3 segundos). 
- *Tiempo con std::launch::deferred*: Aproximadamente 5000 ms. En este modo, la ejecución no es paralela; las tareas se ejecutan de forma secuencial una tras otra en el hilo principal, sumando sus tiempos individuales (2000 ms + 3000 ms). 

La diferencia reside en la política de evaluación. Mientras que *async* fuerza la creación de hilos para la ejecución simultánea, *deferred* pospone la ejecución de la función hasta que se invoca explícitamente el método *.get()* o *.wait()*, realizando una llamada a función convencional (síncrona) en el mismo hilo.

**0.2.3 ¿Qué diferencia hay entre los métodos *wait* y *get* de *std::future*?**

Ambos métodos de *std::future* se utilizan para gestionar la finalización de una tarea asíncrona, pero tienen algunas diferencias clave: El método *wait()* bloquea la ejecución del hilo invocador hasta que la tarea asíncrona termine, sin devolver ningún resultado, por lo que se emplea principalmente con fines de sincronización cuando solo interesa saber que la ejecución ha finalizado; mientras que el método *get()* también bloquea hasta la finalización, pero además recupera y devuelve el valor calculado por la tarea asíncrona. Es importante destacar que *get()* solo puede llamarse una vez por cada objeto future, ya que al hacerlo se mueve el resultado, quedando este inaccesible para llamadas posteriores.

**0.2.4 ¿Qué ventajas ofrece *std::async* frente a *std::thread*?**

La principal ventaja de *std::async* es que ofrece un nivel de abstracción superior orientado a tareas en lugar de a hilos. Sus beneficios clave son: 
- *Gestión de resultados*: *std::async* permite devolver valores directamente mediante objetos future, mientras que *std::thread* requiere mecanismos manuales complejos como std::promise o variables globales. 
- *Propagación de excepciones*: Si la función asíncrona lanza una excepción, *std::async* la captura y la relanza en el hilo principal al llamar a *.get()*, facilitando la depuración. 
- *Gestión del ciclo de vida*: Con *std::thread*, el programador debe decidir manualmente si hacer *join()* o *detach()*; con *std::async*, el sistema gestiona la finalización de forma más automatizada y segura. 


#### **Tarea 0.3: Entrenamiento previo *std::vector***

**0.3.1: ¿Cuál de las dos formas de inicializar el vector y rellenarlo es más eficiente? ¿Por qué?**

La segunda forma (inicializar el vector con un tamaño predefinido: *std::vector<*float*> v2(10000)*) es significativamente más eficiente.
La causa principal radica en la gestión de la memoria dinámica. Cuando se utiliza *push_back()* en un vector sin tamaño definido, el contenedor realiza múltiples realocaciones conforme crece. Cada vez que el vector agota su capacidad actual, el sistema debe:
1. Reservar un nuevo bloque de memoria más grande (normalmente el doble del anterior). 
2. Copiar todos los elementos existentes a la nueva ubicación. 
3. Liberar la memoria antigua.

Este proceso genera un alto coste de CPU y fragmentación de memoria. Al pre-asignar el tamaño, se realiza una única reserva de memoria, eliminando el overhead de copia y permitiendo que el sistema optimice el acceso directo a las posiciones mediante el operador de índice.

**0.3.2: ¿Podría ocurrir algún problema al paralelizar los dos bucles for? ¿Por qué?**

La posibilidad de paralelización depende de la seguridad de hilo (thread safety) de las operaciones realizadas:
- *Primer bucle (push_back)*: No es seguro paralelizarlo. La función *push_back()* no es atómica ni thread-safe; modifica la estructura interna del vector (su tamaño y, potencialmente, su dirección de memoria). Si varios hilos intentan ejecutar *push_back()* simultáneamente, se produciría una condición de carrera crítica que corrompería los punteros internos del vector, provocando probablemente un fallo de segmentación (segmentation fault). 
- *Segundo bucle (v2[i])*: Es totalmente seguro paralelizarlo. Dado que la memoria ya ha sido reservada y el tamaño del vector es fijo, cada hilo trabaja sobre un índice i único y predecible. Al no existir solapamiento entre las posiciones de memoria a las que accede cada hilo (cada uno escribe en su propia "celda"), no hay conflictos de escritura ni dependencias de datos, permitiendo una ejecución paralela perfecta.

### Tarea 1: Paralelización del análisis forense de manipulación de imágenes digitales

#### **Tarea 1.1: Analiza el código e identifica los distintos procesos**

**1. Identificación de procesos**

Tras analizar el flujo de ejecución en main.cc, podemos identificar los siguientes procesos independientes que actúan sobre la imagen original:

1. *Carga de datos*: El proceso load_from_file actúa como el nodo raíz.
2. *Análisis SRM (3x3 y 5x5)*: Dos ejecuciones de filtrado de ruido de alta frecuencia. 
3. *Análisis ELA*: Proceso de análisis de error de nivel que implica recompresión. 
4. *Análisis DCT (Inverso y Directo)*: Dos transformaciones de frecuencia independientes. 
5. *Serialización*: Cinco procesos finales de escritura a disco (save_to_file).

**2. Grafo de dependencias de tareas**

Para que el diseño sea eficiente, se ha mapeado el flujo de datos. Se observa que la imagen original es un recurso de solo lectura. Al no haber escrituras competitivas en la memoria de la imagen de entrada, el grafo muestra un paralelismo de tareas perfecto.

Dado que los cinco procesos de análisis no presentan dependencias de datos entre sí (no hay dependencias de salida ni antidependencias), la estrategia óptima es el uso de programación asíncrona (std::async). Esto permite solapar el tiempo de ejecución de las tareas más cortas (SRM, ELA) con las más largas (DCT).

![captura](capturas/gdt.png)

#### **Tarea 1.2: Analiza el código de cada proceso y el tiempo**

**1. Análisis de Flujo y Granularidad (El proceso DCT)**

El análisis de grano fino revela que el mayor potencial de paralelismo reside en la función *compute_dct*, la cual representa el 68% del tiempo de ejecución secuencial (1230 ms de los 1809 ms totales). Al segmentarse en bloques de 8x8 independientes, se identifica como el candidato principal para una futura optimización de nivel de datos (Data Parallelism).

**2. Puntos paralelizables identificados**

1. Nivel de tarea: Ejecución simultánea de SRM, ELA y DCT
2. Nivel de datos (bucles, identificado como mejora para la segunda prueba):
    - *compute_dct*: Bucle de iteración sobre el vector blocks.
    - *compute_srm*: Bucle de convolución (dentro de utils/image.cc).
    - *compute_ela*: Operaciones de resta matricial y valor absoluto.

![capturas](capturas/terminal1.png)

**3. Resultados del perfilado secuencial (baseline)**

Las mediciones obtenidas en el entorno de ejecución (WSL/Ubuntu) con la imagen de prueba proporcionan la base para el cálculo del Speed-up posterior:

| Proceso | Tiempo Secuencial (ms) | Peso relativo |
|---|---:|---:|
| SRM (3x3 + 5x5) | 76 ms | 4.2% |
| ELA Analysis | 58 ms | 3.2% |
| DCT Inversa (8x8) | 828 ms | 45.7% |
| DCT Directa (8x8) | 402 ms | 22.2% |
| TOTAL (Secuencial) | 1809 ms | 100% |

Análisis de bottleneck: El proceso DCT inverso es el principal limitador del rendimiento. Según la Ley de Amdahl, la aceleración máxima del sistema estará limitada por la fracción secuencial y el tiempo de esta tarea. Al paralelizar el bucle interno de la DCT(segunda prueba), se espera una reducción drástica del tiempo global, ya que es el componente con mayor "intensidad computacional".

Se observa una diferencia entre la suma de los tiempos de computación (1364 ms) y el tiempo total de ejecución secuencial (1809 ms). Estos 445 ms adicionales corresponden al tiempo de entrada/salida (I/O), específicamente a la codificación y escritura en disco de los 5 archivos de salida mediante la función *save_to_file*. Este tiempo se considera parte del 'tiempo de muro' (Wall time) percibido por el usuario.

**Grafo prueba 2:**

![capturas](capturas/g2.png)

Aunque el grafo representa lo ideal, nosotros hemos priorizado la paralelización interna de
DCT mediante OpenMP ya que es, con diferencia, la más costosa (cuello de botella).

#### **Tarea 1.3: Paralelización de la herramienta forense (prueba 1)**

**1. Estrategia de paralelización y justificación**

Para optimizar la aplicación, se ha diseñado una estrategia de paralelismo en dos fases, aunque en esta Prueba 1 nos hemos centrado en el nivel de tareas:

- *Paralelismo de tareas (grano grueso):* Se ha utilizado la librería *<future>* con *std::async* para ejecutar simultáneamente las cinco funciones de análisis (SRM 3x3, SRM 5x5, ELA, DCT Directa y DCT Inversa). Dado que estos algoritmos son independientes entre sí y no presentan dependencias de datos, ejecutarlos en hilos
separados permite que el tiempo total de ejecución dependa únicamente de la tarea más lenta, en lugar de la suma de todas ellas.

- *Identificación del cuello de botella (grano fino)*: Durante la medición de la Prueba 1, se ha confirmado que la DCT es el proceso más costoso, representando aproximadamente el 68% del tiempo total de cómputo (1230 ms de 1809 ms en secuencial). Esto justifica la propuesta de aplicar OpenMP en la siguiente fase (Prueba 2) para paralelizar el bucle de bloques mediante la directiva *#pragma omp parallel for*.

![captura](capturas/gdt.png)

Como se observa en el grafo de la Prueba 1, la aplicación utiliza un paralelismo de tareas asíncronas (grano grueso):
    
- *Solapamiento de algoritmos*: Se ejecutan simultáneamente las cinco funciones de análisis forense, aprovechando que son procesos independientes entre sí.

- *Distribución de carga*: Mientras que las tareas ligeras (SRM/ELA) ocupan sus propios hilos y finalizan rápidamente, las tareas pesadas (DCT directa e inversa) continúan procesándose en paralelo. De esta forma, el tiempo total de ejecución deja de ser la suma de todas las funciones y pasa a estar determinado casi exclusivamente por la duración de la tarea más costosa (DCT inversa).

![capturas](capturas/terminal2Nico.png)

**3. Análisis de rendimiento (speed-up)**

Basado en  las pruebas realizadas en un entorno Linux (WSL), los resultados que hemos obtenido son:

| Algoritmo      | Tiempo Secuencial (ms) | Tiempo Paralelo (ms) | Speed-up (Sp) |
|----------------|------------------------|----------------------|---------------|
| SRM 3x3        | 28 ms                  | 36 ms                | 0.77x         |
| SRM 5x5        | 48 ms                  | 64 ms                | 0.75x         |
| ELA            | 58 ms                  | 64 ms                | 0.9x         |
| DCT Directa    | 402 ms                 | 385 ms               | 1.04x         |
| DCT Inversa    | 828 ms                 | 572 ms               | 1.44x         |
| TIEMPO TOTAL| 1809 ms           | 648 ms           | 2.79x     |

Cálculo del speed-up global:

![capturas](capturas/sp%20global.png)

**Tarea 1.3: Paralelización de la Herramienta Forense - Segunda Fase de Optimización (Modelo Híbrido)**

Tras validar el éxito del paralelismo de tareas en la Prueba 1, se realizó una segunda iteración de mejora centrada en la paralelización de grano fino (Data Parallelism). El objetivo es saturar los recursos de la CPU combinando *std::async* con OpenMP.

Mejoras introducidas:
- *Paralelismo Híbrido*: Mientras que el nivel superior mantiene el lanzamiento concurrente de procesos, el nivel inferior explota el paralelismo de datos en las
funciones con mayor carga computacional.

- Optimización Crítica de la DCT: Se ha implementado la directiva *#pragma omp parallel for* con un *schedule(dynamic)* en el bucle de procesamiento de bloques de la DCT. Dado que la DCT representa el 68% del tiempo de proceso, esta mejora es la que más impacta en el Speed-up global, permitiendo que la tarea más lenta del grafo se ejecute en una fracción del tiempo original.

![capturas](capturas/terminal%20Miguel.png)

**Tabla de Rendimiento: Prueba 2 (Speed-up):**

| Algoritmo      | Tiempo Secuencial (ms) | Tiempo Prueba 2 (ms) | Speed-up Prueba 2 |
|----------------|------------------------|----------------------|-------------------|
| SRM 3x3        | 28 ms                  | 20 ms                | 1.4x             |
| SRM 5x5        | 48 ms                  | 35 ms                | 1.37x             |
| ELA            | 58 ms                  | 28 ms                | 2.07x             |
| DCT Directa    | 402 ms                 | 339 ms               | 1.19x             |
| DCT Inversa    | 828 ms                 | 628 ms               | 1.32x             |
| TIEMPO TOTAL| 1809 ms           | 673 ms           | 2.69x         |

**Análisis de prueba 2:**

En esta segunda prueba se ha implementado un modelo de paralelismo anidado, aplicando OpenMP en el main.cc.
- *Eficiencia en SRM*: La paralelización a nivel de bucle ha reducido SRM 3x3 de 28 ms a 20 ms (1.40x) y SRM 5x5 de 48 ms a 35 ms (1.37x). Se debe añadir que la tarea más beneficiada es ELA, que pasa de 58 ms a 28 ms logrando el mayor Speed-up individual con un 2.07x.

- *Balanceo de Carga*: El tiempo total sigue dominado por la DCT Inversa (628 ms), que actúa como cuello de botella principal. Las tareas más cortas (SRM, ELA) finalizan mucho antes, dejando sus hilos ociosos y limitando la eficiencia global del paralelismo.

- *Contención de Recursos*: El tiempo total de la Prueba 2 (673 ms) es ligeramente superior al de la Prueba 1 (648 ms), lo que indica que el paralelismo anidado introduce una sobrecarga adicional. La competencia entre hilos por el ancho de banda de memoria y los costes de sincronización de OpenMP superan en este caso la ganancia obtenida a nivel de tarea, resultando en un Speed-up total de 2.69x frente al 2.79x de la Prueba 1.

- *Speed-up Total Calculado (Prueba 2)*:

![capturas](capturas/sp%202.png)

**Respuestas a las cuestiones de rendimiento**

- *¿Se obtiene un mejor rendimiento paralelizando todas las partes posibles?*

En general sí, ambas pruebas reducen significativamente el tiempo total respecto al secuencial (de 1809 ms a 648 ms en la Prueba 1 y 673 ms en la Prueba 2). Sin embargo, el rendimiento no escala linealmente debido a la Ley de Amdahl: tareas como la DCT Inversa siguen siendo un cuello de botella, y la sobrecarga de sincronización de hilos limita la ganancia máxima obtenible. Además, la Prueba 2 muestra que añadir más paralelismo no siempre mejora el rendimiento, ya que su tiempo total es ligeramente superior al de la Prueba 1 debido a la contención de recursos y los costes de sincronización de OpenMP.

- *¿Se degrada el rendimiento al paralelizar ciertas partes?*

Sí, se observa una ligera degradación en funciones como SRM y ELA, cuyos tiempos de ejecución individuales aumentan ligeramente en comparación con la versión secuencial.

- *¿A qué creéis que se debe esa degradación?*

Se debe principalmente a la competencia por recursos (contención de memoria caché y ancho de banda del bus de datos) al ejecutarse simultáneamente con la DCT, además del overhead o sobrecoste que implica la creación y gestión de hilos (context switching).

- *¿Cuál es la mejor estrategia de paralelización?*

La estrategia híbrida: usar paralelismo de tareas (std::async) para solapar algoritmos independientes y paralelismo de datos (OpenMP) para triturar el cuello de botella computacional de la DCT. Esta combinación maximiza la ocupación de todos los núcleos de la CPU.

**Sobre el paralelismo explotado:**

- *Tipos de paralelismo usado*: Se emplea un paralelismo híbrido. A nivel externo, paralelismo de tareas (ejecución de diferentes algoritmos de forma independiente) y a nivel interno en las funciones DCT, paralelismo de datos (reparto de los bloques de píxeles entre hilos).

- *Modo de programación paralela*: Se basa en un modelo de memoria compartida. Todos los hilos (tanto los creados por *std::async* como por *OpenMP*) acceden al mismo espacio de direccionamiento de memoria para leer la imagen original cargada en el sistema.

- *Alternativas de comunicación*: La comunicación es implícita. Al trabajar en memoria compartida, los hilos se comunican a través de las variables del programa y la estructura de la imagen, sin necesidad de paso de mensajes explícitos (como ocurriría en MPI). La sincronización final se gestiona mediante el mecanismo de futuros (*std::future::get()*).

- *Estilo de programación paralela*: Se utiliza un estilo de paralelismo multihilo (Multithreading). Se emplea un enfoque de paralelismo estructurado con OpenMP (mediante directivas de compilación) y un enfoque basado en tareas (Task-based) con la librería estándar de C++ (*std::async*).

- *Tipo de estructura paralela del programa*: Sigue una estructura de Master-Worker (o Maestro-Trabajador). El hilo principal (Master) actúa como coordinador, lanzando las tareas asíncronas y los hilos de OpenMP (Workers) para que realicen el cómputo intensivo, recolectando finalmente los resultados para el almacenamiento en disco.


**Sobre los resultados de las pruebas realizadas y su contexto:**

- *Caracterización del sistema*: Las pruebas se han realizado en un procesador Intel(R) Core(TM) i5-13420H (13ª generación). Se trata de una arquitectura con 6 núcleos físicos y un total de 12 hilos lógicos (CPUs). El sistema de memoria caché se organiza en tres niveles:
    - L1: 288 KiB (instrucciones) y 192 KiB (datos).
    - L2: 7.5 MiB.
    - L3 (compartida): 12 MiB, lo que facilita la comunicación rápida entre hilos en memoria compartida. El programa se ejecuta en un entorno Linux mediante el hipervisor de Microsoft (WSL).

- *Significado de la palabra ht*: En la salida del comando *lscpu* (dentro de la sección de Flags), la sigla ht hace referencia a Hyper-Threading. Es una tecnología de Intel que permite que cada núcleo físico ejecute dos hilos de procesamiento simultáneos, mejorando el rendimiento en aplicaciones multihilo.

- *Gráficas*: Ganancia en velocidad (speed-up) en función del número de unidades de cómputo (threads en este caso) y ganancia en velocidad (speed-up) en función de los parámetros que modifiquen el tamaño del problema:

![capturas](capturas/graf1.png)

![capturas](capturas/graf2.png)

Se observa que la ganancia de velocidad aumenta de forma notable hasta los 4-6 hilos (coincidiendo con tus núcleos físicos). A partir de ahí, la curva se aplanan. Esto se debe al overhead de gestión de hilos y a que el Hyper-Threading (hilos lógicos) no ofrece el mismo rendimiento que un núcleo físico real.

Además, la ganancia en velocidad (speed-up) es directamente proporcional al tamaño del problema (resolución de la imagen). En problemas de baja talla, el sobrecoste (overhead) de creación y sincronización de hilos penaliza el rendimiento. Sin embargo, en tallas grandes, la carga computacional de los filtros SRM y la DCT permite amortizar dicho coste, logrando que el sistema se acerque más a la aceleración lineal y aproveche mejor los 12 hilos lógicos de la CPU.

Ganancia en Velocidad Máxima Teórica (Ley de Amdahl):

La ganancia máxima teórica no depende solo del número de núcleos, sino de la fracción del código que es secuencial (carga de imagen desde disco, gestión de archivos y escritura de resultados).

Según la Ley de Amdahl, si definimos P como la fracción paralelizable del programa:

![capturas](capturas/LdA.png)

Si estimamos que el 30% del programa es secuencial (E/S de archivos), la ganancia máxima teórica sería de 3.33x. El resultado obtenido de 1,78x es un valor excelente y realista, ya que se encuentra cerca del límite práctico impuesto por la arquitectura de memoria y el sistema de archivos del ordenador.

**Responda de forma justificada: ¿Cuál es la implementación más eficiente de las 2?**

La implementación más eficiente es la versión paralela híbrida (Prueba 1). Ya que esta reduce el tiempo de ejecución de 1809 ms a 648 ms, lo que supone una mejora del 64% y un Speed-up de 2.79x, aprovechando los 16 hilos lógicos disponibles frente al único núcleo de la versión secuencial, que solo utilizaba el 6.25% de la capacidad del procesador. Además, reduce significativamente el cuello de botella principal, la DCT Inversa, de 828 ms a 572 ms mediante el reparto de carga de OpenMP. Por último, a diferencia de la versión secuencial cuyo rendimiento es estático, la versión paralela escala su eficiencia proporcionalmente a la resolución de la imagen y al número de núcleos disponibles.